In [1]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms

import numpy as np
import faiss

from PIL import Image
from pathlib import Path

import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [2]:
# Load FAISS index
index = faiss.read_index(
    "../faiss_index/plant_disease.index"
)

# Load labels
labels = np.load(
    "../embeddings/labels.npy",
    allow_pickle=True
)

# Load image paths
paths = np.load(
    "../embeddings/paths.npy",
    allow_pickle=True
)

print("Vectors in FAISS:", index.ntotal)
print("Labels:", labels.shape)
print("Paths:", paths.shape)

Vectors in FAISS: 49984
Labels: (49984,)
Paths: (49984,)


In [4]:
resnet = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

feature_extractor = torch.nn.Sequential(
    *list(resnet.children())[:-1]
)

feature_extractor.eval();

In [5]:
import random

test_folder = Path(
    "../dataset/Apple/Test/Apple Scab"
)

test_images = list(
    test_folder.glob("*")
)

query_image = random.choice(
    test_images
)

print(query_image)

..\dataset\Apple\Test\Apple Scab\d608ac35-eaf8-43e2-b429-f9b433126023___FREC_Scab 3386_270deg.JPG


In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def get_query_embedding(image_path):

    image = Image.open(image_path).convert("RGB")

    image = transform(image).unsqueeze(0)

    with torch.no_grad():
        embedding = feature_extractor(image)

    embedding = embedding.squeeze().numpy().astype("float32")

    return embedding

In [7]:
query_embedding = get_query_embedding(query_image)

print(query_embedding.shape)

(2048,)


In [18]:
k = 20

distances, indices = index.search(
    query_embedding.reshape(1, -1),
    k
)

print("Distances:")
print(distances)

print("\nIndices:")
print(indices)

Distances:
[[70.11014  76.827866 77.49875  78.56279  78.99107  79.01991  79.864975
  79.89841  80.08641  80.13194  80.8392   81.034775 81.204636 81.49985
  81.51131  81.518906 81.7511   81.80209  81.89943  81.91669 ]]

Indices:
[[ 1175 41545  8044  8369 40409 40408 39816 24208 42501 34092 35130 24207
  41609 34214  1047 41245 41228   795   720  8370]]


In [19]:
for i in range(k):

    idx = indices[0][i]

    print(f"Rank {i+1}")
    print("Distance :", distances[0][i])
    print("Label    :", labels[idx])
    print("Path     :", paths[idx])
    print("-" * 50)

Rank 1
Distance : 70.11014
Label    : Apple_Apple Scab
Path     : ..\dataset\Apple\Train\Apple Scab\98e6e72f-092d-41dc-93e1-22bfc1ea2abd___FREC_Scab 2953_90deg.JPG
--------------------------------------------------
Rank 2
Distance : 76.827866
Label    : Strawberry_Leaf Scorch
Path     : ..\dataset\Strawberry\Train\Leaf Scorch\7014ff4d-61a0-427b-8ef0-390eb0ed0120___RS_L.Scorch 1069.JPG
--------------------------------------------------
Rank 3
Distance : 77.49875
Label    : Bell Pepper_Bacterial Spot
Path     : ..\dataset\Bell Pepper\Train\Bacterial Spot\2f9b46c7-fa77-4196-9ed1-f3f316182340___JR_B.Spot 8943.JPG
--------------------------------------------------
Rank 4
Distance : 78.56279
Label    : Bell Pepper_Bacterial Spot
Path     : ..\dataset\Bell Pepper\Train\Bacterial Spot\59b6bff9-4aca-4bf4-a129-21179e9cebae___JR_B.Spot 9069_180deg.JPG
--------------------------------------------------
Rank 5
Distance : 78.99107
Label    : Strawberry_Healthy
Path     : ..\dataset\Strawberry\Train\

In [20]:
from collections import Counter

retrieved_labels = [
    labels[idx]
    for idx in indices[0]
]

counts = Counter(retrieved_labels)

for label, count in counts.items():
    print(label, ":", count)

Apple_Apple Scab : 4
Strawberry_Leaf Scorch : 5
Bell Pepper_Bacterial Spot : 3
Strawberry_Healthy : 3
Grape_Black Rot : 2
Potato_Early Blight : 3


In [15]:
embeddings = np.load("../embeddings/embeddings.npy")

print("Database dtype :", embeddings.dtype)
print("Query dtype    :", query_embedding.dtype)

print("Database shape :", embeddings.shape)
print("Query shape    :", query_embedding.shape)

Database dtype : float32
Query dtype    : float32
Database shape : (49984, 2048)
Query shape    : (2048,)


In [16]:
print("Query norm:", np.linalg.norm(query_embedding))

Query norm: 12.769502


In [17]:
print("Database norm:", np.linalg.norm(embeddings[0]))

Database norm: 12.344364
